## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: https://planetarycomputer.microsoft.com/dataset/terraclimate#overview 

In [17]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os
import time

<h2>Extracting TerraClimate Data Using API Calls</h2> 
<p align="justify"> The API-based method allows us to efficiently access <b>TerraClimate</b> data for specific regions and time periods through the <a href="https://planetarycomputer.microsoft.com/">Microsoft Planetary Computer</a>, ensuring scalability and reproducibility of the process. </p> 

<p align="justify"> 
<b>🔧 OPTIMIZATION 10 UPDATE:</b> This notebook now extracts <b>5 critical climate variables</b> that drive water chemistry:
</p>

<ul>
  <li><b>pet</b> - Potential Evapotranspiration (atmospheric water demand)</li>
  <li><b>ppt</b> - Precipitation (rainfall - controls dilution/concentration)</li>
  <li><b>soil</b> - Soil Moisture (controls mineral leaching)</li>
  <li><b>tmax</b> - Maximum Temperature (affects reaction rates)</li>
  <li><b>tmin</b> - Minimum Temperature (affects biological processes)</li>
</ul>

<p align="justify"> These variables represent the <b>universal physical drivers</b> of water quality that transfer across different geographic locations, unlike site-specific optical features. The extraction will produce files with all 5 variables: <code>terraclimate_features_training_EXTENDED.csv</code> and <code>terraclimate_features_validation_EXTENDED.csv</code>. </p>

<h3>Loading and Mapping TerraClimate Data:</h3>

<p>This section demonstrates how <b>TerraClimate climate variables</b>, such as <b>Potential Evapotranspiration (PET)</b>, are loaded and mapped to sampling locations:</p>

<ul>
  <li>The <b>load_terraclimate_dataset</b> function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.</li>
  <li>The <b>filterg</b> function filters the dataset for the desired time range (2011–2015) and spatial extent corresponding to the study region. The resulting data is converted to a pandas DataFrame with standardized column names.</li>
  <li>The <b>assign_nearest_climate</b> function maps each sampling location to its <b>nearest TerraClimate grid point</b> using a KD-tree and assigns the climate variable values corresponding to the closest time stamp.</li>
</ul>

<p>This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.</p>


In [18]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [19]:
# --- Filtering function (optimized: spatial subset + retries + diagnostics) ---
def filterg(ds, var):
    # Time range selection
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    # Spatial bounds for the study region (apply BEFORE converting to dataframe)
    lat_min, lat_max = -35.18, -21.72
    lon_min, lon_max = 14.97, 32.79

    # Detect coordinate names (common variants)
    coord_candidates = list(ds_2011_2015.coords.keys())
    lat_name = None
    lon_name = None
    for c in coord_candidates:
        if c.lower() in ('lat','latitude') and lat_name is None:
            lat_name = c
        if c.lower() in ('lon','longitude') and lon_name is None:
            lon_name = c

    if lat_name is None or lon_name is None:
        print(f"Warning: could not detect lat/lon coordinate names in dataset coords: {coord_candidates}; falling back to full read")
        ds_subset = ds_2011_2015
    else:
        # Decide slice ordering based on coordinate direction
        lat_vals = ds_2011_2015[lat_name].values
        lon_vals = ds_2011_2015[lon_name].values
        lat_asc = lat_vals[0] < lat_vals[-1]
        lon_asc = lon_vals[0] < lon_vals[-1]

        if lat_asc:
            lat_slice = slice(lat_min, lat_max)
        else:
            lat_slice = slice(lat_max, lat_min)
        if lon_asc:
            lon_slice = slice(lon_min, lon_max)
        else:
            lon_slice = slice(lon_max, lon_min)

        try:
            ds_subset = ds_2011_2015.sel({lat_name: lat_slice, lon_name: lon_slice})
        except Exception as e:
            print(f"Warning: spatial sel failed ({e}); falling back to full read")
            ds_subset = ds_2011_2015

    # Diagnostics: print coordinate ranges and subset sizes
    try:
        coords_info = {
            'lat_name': lat_name,
            'lon_name': lon_name,
            'lat_range': (float(ds_2011_2015[lat_name].values.min()), float(ds_2011_2015[lat_name].values.max())) if lat_name else None,
            'lon_range': (float(ds_2011_2015[lon_name].values.min()), float(ds_2011_2015[lon_name].values.max())) if lon_name else None,
            'subset_sizes': dict(ds_subset.sizes) if hasattr(ds_subset, 'sizes') else None,
        }
        print('Diagnostics:', coords_info)
    except Exception:
        pass

    df_var_append = []
    for i in tqdm(range(len(ds_subset.time))):
        # Add simple retry/backoff to handle transient network timeouts
        attempts = 0
        max_attempts = 3
        df_var = pd.DataFrame()
        while attempts < max_attempts:
            try:
                df_var = ds_subset.isel(time=i).to_dataframe().reset_index()
                break
            except Exception as e:
                attempts += 1
                wait = 5 * attempts
                print(f"Warning: read attempt {attempts} failed (sleep {wait}s). Error: {e}")
                time.sleep(wait)
                if attempts >= max_attempts:
                    # skip this timestep if it cannot be read
                    df_var = pd.DataFrame(columns=['lat', 'lon', 'time', var])
                    break

        # If df_var has the full grid, filter again to ensure bounds
        if not df_var.empty and 'lat' in df_var.columns and 'lon' in df_var.columns:
            df_var_filter = df_var[
                (df_var['lat'] >= lat_min) & (df_var['lat'] <= lat_max) &
                (df_var['lon'] >= lon_min) & (df_var['lon'] <= lon_max)
            ]
        else:
            df_var_filter = df_var

        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True) if df_var_append else pd.DataFrame()
    print(f"Filtering for {var} completed")

    if not df_var_final.empty:
        df_var_final['time'] = df_var_final['time'].astype(str)

        # Column mapping
        col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
        df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final

In [20]:
# --- Climate variable assignment function (improved robustness) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    Robustness improvements:
    - Accepts climate_df with either 'lat'/'lon' or 'Latitude'/'Longitude' column names.
    - If the filtered climate_df is empty, return NaNs for all samples instead of raising.
    """
    # Defensive: if climate_df is None, return NaNs
    if climate_df is None:
        print(f"Warning: climate_df is None for '{var_name}'; returning NaNs")
        return pd.DataFrame({var_name: [np.nan] * len(sa_df)})

    # Normalize latitude/longitude/time column names if needed
    cols_lower = [c.lower() for c in climate_df.columns]
    rename_map = {}
    for c in climate_df.columns:
        if c.lower() == 'lat':
            rename_map[c] = 'Latitude'
        if c.lower() == 'lon':
            rename_map[c] = 'Longitude'
        if c.lower() == 'time' and 'Sample Date' not in climate_df.columns:
            rename_map[c] = 'Sample Date'
    if rename_map:
        climate_df = climate_df.rename(columns=rename_map)

    # If climate_df is empty after filtering, return NaNs for all samples
    if climate_df.empty:
        print(f"Warning: filtered climate dataframe for '{var_name}' is empty; returning NaNs")
        return pd.DataFrame({var_name: [np.nan] * len(sa_df)})

    # Verify required columns exist now
    if not {'Latitude','Longitude'}.issubset(set(climate_df.columns)):
        raise KeyError(f"climate_df must contain latitude/longitude columns. Found: {list(climate_df.columns)}")

    # Prepare coordinates for KD-tree matching
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    return output_df

In [21]:
# --- QUIET version of climate assignment (no tqdm spam for validation) ---
def assign_nearest_climate_QUIET(sa_df, climate_df, var_name):
    """Quiet version - prints progress every 50 rows instead of using tqdm"""
    if climate_df is None or climate_df.empty:
        print(f"    ⚠️  Warning: No climate data for '{var_name}'; returning NaNs")
        return pd.DataFrame({var_name: [np.nan] * len(sa_df)})
    
    # Normalize column names
    rename_map = {}
    for c in climate_df.columns:
        if c.lower() == 'lat': rename_map[c] = 'Latitude'
        if c.lower() == 'lon': rename_map[c] = 'Longitude'
        if c.lower() == 'time' and 'Sample Date' not in climate_df.columns:
            rename_map[c] = 'Sample Date'
    if rename_map:
        climate_df = climate_df.rename(columns=rename_map)
    
    # Build KD tree for spatial matching
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]
    
    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')
    
    climate_values = []
    total = len(sa_df)
    
    for i in range(total):
        # Quiet progress (only every 50 rows)
        if i > 0 and i % 50 == 0:
            print(f"    ... {i}/{total} rows processed")
        
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']
        
        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]
        
        if subset.empty:
            climate_values.append(np.nan)
        else:
            nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
            climate_values.append(subset.loc[nearest_idx, var_name])
    
    return pd.DataFrame({var_name: climate_values})

### Extracting features for the training and validation datasets

**📋 EXTRACTION WORKFLOW:**

This section contains **5 separate cells** - one for each climate variable. Each cell performs:
1. Load TerraClimate dataset (with fresh authentication)
2. Extract variable for **training** locations
3. Extract variable for **validation** locations
4. Store in memory for final combination

**✅ BENEFITS:**
- **No timeout errors**: Each variable runs independently
- **Fresh authentication**: New dataset connection per variable prevents auth expiration
- **Resume capability**: If one fails, re-run only that cell
- **No memory overload**: Each extraction completes before the next

**🚀 VARIABLES TO EXTRACT:**
1. **PET** - Potential Evapotranspiration
2. **PPT** - Precipitation (mm)
3. **SOIL** - Soil Moisture (mm)
4. **TMAX** - Maximum Temperature (°C)
5. **TMIN** - Minimum Temperature (°C)

**⏱️ ESTIMATED TIME:** ~5-10 minutes per variable (25-50 minutes total)

After all 5 variables are extracted, run the final **combine and save** cells to create the output files.

In [22]:
Water_Quality_df=pd.read_csv('water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [23]:
Water_Quality_df.shape

(9319, 6)

In [24]:
Validation_df=pd.read_csv('submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [10]:
Validation_df.shape

(200, 6)

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# EXTRACT VARIABLE 1: PET (Potential Evapotranspiration)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  EXTRACTING PET - Potential Evapotranspiration")
print("═" * 80)

# Load TerraClimate dataset
print("\n[1/3] Loading TerraClimate dataset...")
ds_pet = load_terraclimate_dataset()
print("      ✓ Dataset loaded successfully")

# Extract PET for training
print("\n[2/3] Extracting PET for training locations...")
tc_pet_training = filterg(ds_pet, 'pet')
pet_training_mapped = assign_nearest_climate(Water_Quality_df, tc_pet_training, 'pet')
print(f"      ✓ PET training extracted: {len(pet_training_mapped)} rows")

# Extract PET for validation (QUIET mode)
print("\n[3/3] Extracting PET for validation locations...")
tc_pet_validation = filterg(ds_pet, 'pet')
pet_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_pet_validation, 'pet')
print(f"      ✓ PET validation extracted: {len(pet_validation_mapped)} rows")

print("\n✅ PET EXTRACTION COMPLETE!")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
  EXTRACTING PET - Potential Evapotranspiration
════════════════════════════════════════════════════════════════════════════════

[1/3] Loading TerraClimate dataset...
      ✓ Dataset loaded successfully

[2/3] Extracting PET for training locations...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [11:46<00:00, 11.78s/it]


Filtering for pet completed


Mapping PET values: 100%|██████████| 9319/9319 [05:32<00:00, 28.07it/s]


      ✓ PET training extracted: 9319 rows

[3/3] Extracting PET for validation locations...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [16:30<00:00, 16.50s/it]


Filtering for pet completed
    ... 50/200 rows processed
    ... 100/200 rows processed
    ... 150/200 rows processed
      ✓ PET validation extracted: 200 rows

✅ PET EXTRACTION COMPLETE!
════════════════════════════════════════════════════════════════════════════════


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXTRACT VARIABLE 2: PPT (Precipitation)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  EXTRACTING PPT - Precipitation (mm)")
print("═" * 80)

# Load TerraClimate dataset
print("\n[1/3] Loading TerraClimate dataset...")
ds_ppt = load_terraclimate_dataset()
print("      ✓ Dataset loaded successfully")

# Extract PPT for training
print("\n[2/3] Extracting PPT for training locations...")
tc_ppt_training = filterg(ds_ppt, 'ppt')
ppt_training_mapped = assign_nearest_climate(Water_Quality_df, tc_ppt_training, 'ppt')
print(f"      ✓ PPT training extracted: {len(ppt_training_mapped)} rows")

# Extract PPT for validation (QUIET mode)
print("\n[3/3] Extracting PPT for validation locations...")
tc_ppt_validation = filterg(ds_ppt, 'ppt')
ppt_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_ppt_validation, 'ppt')
print(f"      ✓ PPT validation extracted: {len(ppt_validation_mapped)} rows")

print("\n✅ PPT EXTRACTION COMPLETE!")
print("═" * 80)

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# 🔄 RECOVERY: Extract PPT validation ONLY (if training already done)
# ══════════════════════════════════════════════════════════════════════════════
# Use this cell if PPT training succeeded but validation timed out
# This loads a fresh dataset with new authentication token

print("═" * 80)
print("  🔄 RECOVERY MODE: Extracting PPT validation only")
print("═" * 80)

# Load fresh TerraClimate dataset (new auth token)
print("\n[1/2] Loading TerraClimate dataset with fresh authentication...")
ds_ppt_recovery = load_terraclimate_dataset()
print("      ✓ Dataset loaded with new auth token")

# Extract PPT for validation only (QUIET mode)
print("\n[2/2] Extracting PPT for validation locations...")
tc_ppt_validation = filterg(ds_ppt_recovery, 'ppt')
ppt_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_ppt_validation, 'ppt')
print(f"      ✓ PPT validation extracted: {len(ppt_validation_mapped)} rows")

print("\n✅ PPT VALIDATION RECOVERY COMPLETE!")
print(f"   Training data preserved: {len(ppt_training_mapped)} rows")
print(f"   Validation data created: {len(ppt_validation_mapped)} rows")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
  🔄 RECOVERY MODE: Extracting PPT validation only
════════════════════════════════════════════════════════════════════════════════

[1/2] Loading TerraClimate dataset with fresh authentication...
      ✓ Dataset loaded with new auth token

[2/2] Extracting PPT for validation locations...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [12:25<00:00, 12.43s/it]


Filtering for ppt completed
    ... 50/200 rows processed
    ... 100/200 rows processed
    ... 150/200 rows processed
      ✓ PPT validation extracted: 200 rows

✅ PPT VALIDATION RECOVERY COMPLETE!
   Training data preserved: 9319 rows
   Validation data created: 200 rows
════════════════════════════════════════════════════════════════════════════════


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXTRACT VARIABLE 3: SOIL (Soil Moisture)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  EXTRACTING SOIL - Soil Moisture (mm)")
print("═" * 80)

# Load TerraClimate dataset
print("\n[1/3] Loading TerraClimate dataset...")
ds_soil = load_terraclimate_dataset()
print("      ✓ Dataset loaded successfully")

# Extract SOIL for training
print("\n[2/3] Extracting SOIL for training locations...")
tc_soil_training = filterg(ds_soil, 'soil')
soil_training_mapped = assign_nearest_climate(Water_Quality_df, tc_soil_training, 'soil')
print(f"      ✓ SOIL training extracted: {len(soil_training_mapped)} rows")

# Extract SOIL for validation (QUIET mode)
print("\n[3/3] Extracting SOIL for validation locations...")
tc_soil_validation = filterg(ds_soil, 'soil')
soil_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_soil_validation, 'soil')
print(f"      ✓ SOIL validation extracted: {len(soil_validation_mapped)} rows")

print("\n✅ SOIL EXTRACTION COMPLETE!")
print("═" * 80)

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# 🔄 RECOVERY: Extract SOIL validation ONLY (if training already done)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  🔄 RECOVERY MODE: Extracting SOIL validation only")
print("═" * 80)

print("\n[1/2] Loading TerraClimate dataset with fresh authentication...")
ds_soil_recovery = load_terraclimate_dataset()
print("      ✓ Dataset loaded with new auth token")

print("\n[2/2] Extracting SOIL for validation locations...")
tc_soil_validation = filterg(ds_soil_recovery, 'soil')
soil_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_soil_validation, 'soil')
print(f"      ✓ SOIL validation extracted: {len(soil_validation_mapped)} rows")

print("\n✅ SOIL VALIDATION RECOVERY COMPLETE!")
print(f"   Training data preserved: {len(soil_training_mapped)} rows")
print(f"   Validation data created: {len(soil_validation_mapped)} rows")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
  🔄 RECOVERY MODE: Extracting SOIL validation only
════════════════════════════════════════════════════════════════════════════════

[1/2] Loading TerraClimate dataset with fresh authentication...
      ✓ Dataset loaded with new auth token

[2/2] Extracting SOIL for validation locations...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [06:18<00:00,  6.30s/it]


Filtering for soil completed
    ... 50/200 rows processed
    ... 100/200 rows processed
    ... 150/200 rows processed
      ✓ SOIL validation extracted: 200 rows

✅ SOIL VALIDATION RECOVERY COMPLETE!
   Training data preserved: 9319 rows
   Validation data created: 200 rows
════════════════════════════════════════════════════════════════════════════════


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# EXTRACT VARIABLE 4: TMAX (Maximum Temperature)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  EXTRACTING TMAX - Maximum Temperature (°C)")
print("═" * 80)

# Load TerraClimate dataset
print("\n[1/3] Loading TerraClimate dataset...")
ds_tmax = load_terraclimate_dataset()
print("      ✓ Dataset loaded successfully")

# Extract TMAX for training
print("\n[2/3] Extracting TMAX for training locations...")
tc_tmax_training = filterg(ds_tmax, 'tmax')
tmax_training_mapped = assign_nearest_climate(Water_Quality_df, tc_tmax_training, 'tmax')
print(f"      ✓ TMAX training extracted: {len(tmax_training_mapped)} rows")

# Extract TMAX for validation (QUIET mode)
print("\n[3/3] Extracting TMAX for validation locations...")
tc_tmax_validation = filterg(ds_tmax, 'tmax')
tmax_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_tmax_validation, 'tmax')
print(f"      ✓ TMAX validation extracted: {len(tmax_validation_mapped)} rows")

print("\n✅ TMAX EXTRACTION COMPLETE!")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
  EXTRACTING TMAX - Maximum Temperature (°C)
════════════════════════════════════════════════════════════════════════════════

[1/3] Loading TerraClimate dataset...
      ✓ Dataset loaded successfully

[2/3] Extracting TMAX for training locations...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [08:09<00:00,  8.16s/it]


Filtering for tmax completed


Mapping TMAX values: 100%|██████████| 9319/9319 [03:03<00:00, 50.69it/s]


      ✓ TMAX training extracted: 9319 rows

[3/3] Extracting TMAX for validation locations...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [09:01<00:00,  9.02s/it]


Filtering for tmax completed
    ... 50/200 rows processed
    ... 100/200 rows processed
    ... 150/200 rows processed
      ✓ TMAX validation extracted: 200 rows

✅ TMAX EXTRACTION COMPLETE!
════════════════════════════════════════════════════════════════════════════════


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 🔄 RECOVERY: Extract TMAX validation ONLY (if training already done)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  🔄 RECOVERY MODE: Extracting TMAX validation only")
print("═" * 80)

print("\n[1/2] Loading TerraClimate dataset with fresh authentication...")
ds_tmax_recovery = load_terraclimate_dataset()
print("      ✓ Dataset loaded with new auth token")

print("\n[2/2] Extracting TMAX for validation locations...")
tc_tmax_validation = filterg(ds_tmax_recovery, 'tmax')
tmax_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_tmax_validation, 'tmax')
print(f"      ✓ TMAX validation extracted: {len(tmax_validation_mapped)} rows")

print("\n✅ TMAX VALIDATION RECOVERY COMPLETE!")
print(f"   Training data preserved: {len(tmax_training_mapped)} rows")
print(f"   Validation data created: {len(tmax_validation_mapped)} rows")
print("═" * 80)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXTRACT VARIABLE 5: TMIN (Minimum Temperature)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  EXTRACTING TMIN - Minimum Temperature (°C)")
print("═" * 80)

# Load TerraClimate dataset
print("\n[1/3] Loading TerraClimate dataset...")
ds_tmin = load_terraclimate_dataset()
print("      ✓ Dataset loaded successfully")

# Extract TMIN for training
print("\n[2/3] Extracting TMIN for training locations...")
tc_tmin_training = filterg(ds_tmin, 'tmin')
tmin_training_mapped = assign_nearest_climate(Water_Quality_df, tc_tmin_training, 'tmin')
print(f"      ✓ TMIN training extracted: {len(tmin_training_mapped)} rows")

# Extract TMIN for validation (QUIET mode)
print("\n[3/3] Extracting TMIN for validation locations...")
tc_tmin_validation = filterg(ds_tmin, 'tmin')
tmin_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_tmin_validation, 'tmin')
print(f"      ✓ TMIN validation extracted: {len(tmin_validation_mapped)} rows")

print("\n✅ TMIN EXTRACTION COMPLETE!")
print("═" * 80)

In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# 🔄 RECOVERY: Extract TMIN validation ONLY (if training already done)
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 80)
print("  🔄 RECOVERY MODE: Extracting TMIN validation only")
print("═" * 80)

print("\n[1/2] Loading TerraClimate dataset with fresh authentication...")
ds_tmin_recovery = load_terraclimate_dataset()
print("      ✓ Dataset loaded with new auth token")

print("\n[2/2] Extracting TMIN for validation locations...")
tc_tmin_validation = filterg(ds_tmin_recovery, 'tmin')
tmin_validation_mapped = assign_nearest_climate_QUIET(Validation_df, tc_tmin_validation, 'tmin')
print(f"      ✓ TMIN validation extracted: {len(tmin_validation_mapped)} rows")

print("\n✅ TMIN VALIDATION RECOVERY COMPLETE!")
print(f"   Training data preserved: {len(tmin_training_mapped)} rows")
print(f"   Validation data created: {len(tmin_validation_mapped)} rows")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
  🔄 RECOVERY MODE: Extracting TMIN validation only
════════════════════════════════════════════════════════════════════════════════

[1/2] Loading TerraClimate dataset with fresh authentication...
      ✓ Dataset loaded with new auth token

[2/2] Extracting TMIN for validation locations...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [04:22<00:00,  4.37s/it]


Filtering for tmin completed
    ... 50/200 rows processed
    ... 100/200 rows processed
    ... 150/200 rows processed
      ✓ TMIN validation extracted: 200 rows

✅ TMIN VALIDATION RECOVERY COMPLETE!
   Training data preserved: 9319 rows
   Validation data created: 200 rows
════════════════════════════════════════════════════════════════════════════════


In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# COMBINE ALL VARIABLES AND SAVE TRAINING FILE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 80)
print("  COMBINING ALL VARIABLES - TRAINING DATASET")
print("═" * 80)

# Combine all training variables
print("\n[1/3] Combining all variables into single DataFrame...")
Terraclimate_training_df = pd.concat([
    pet_training_mapped,
    ppt_training_mapped,
    soil_training_mapped,
    tmax_training_mapped,
    tmin_training_mapped
], axis=1)

print(f"      ✓ Combined shape: {Terraclimate_training_df.shape}")
print(f"      ✓ Columns: {list(Terraclimate_training_df.columns)}")

# Add location and date information
print("\n[2/3] Adding location and date columns...")
Terraclimate_training_df['Latitude'] = Water_Quality_df['Latitude'].values
Terraclimate_training_df['Longitude'] = Water_Quality_df['Longitude'].values
Terraclimate_training_df['Sample Date'] = Water_Quality_df['Sample Date'].values

# Reorder columns: Location, Date, then all climate variables
column_order = ['Latitude', 'Longitude', 'Sample Date', 'pet', 'ppt', 'soil', 'tmax', 'tmin']
Terraclimate_training_df = Terraclimate_training_df[column_order]

# Save EXTENDED file for Optimization 10
output_file = 'terraclimate_features_training_EXTENDED.csv'
Terraclimate_training_df.to_csv(output_file, index=False)

print(f"      ✓ Reordered columns: {list(Terraclimate_training_df.columns)}")
print(f"\n[3/3] ✅ TRAINING FILE SAVED: {output_file}")
print(f"      Shape: {Terraclimate_training_df.shape}")
print(f"      Size: {os.path.getsize(output_file) / 1_000_000:.2f} MB")

# Also save legacy single-variable file for backward compatibility
Terraclimate_training_df[['Latitude', 'Longitude', 'Sample Date', 'pet']].to_csv(
    'terraclimate_features_training.csv', index=False
)
print(f"      ✓ Legacy file also saved: terraclimate_features_training.csv (pet only)")
print("═" * 80)


════════════════════════════════════════════════════════════════════════════════
  COMBINING ALL VARIABLES - TRAINING DATASET
════════════════════════════════════════════════════════════════════════════════

[1/3] Combining all variables into single DataFrame...
      ✓ Combined shape: (9319, 5)
      ✓ Columns: ['pet', 'ppt', 'soil', 'tmax', 'tmin']

[2/3] Adding location and date columns...
      ✓ Reordered columns: ['Latitude', 'Longitude', 'Sample Date', 'pet', 'ppt', 'soil', 'tmax', 'tmin']

[3/3] ✅ TRAINING FILE SAVED: terraclimate_features_training_EXTENDED.csv
      Shape: (9319, 8)
      Size: 0.63 MB
      ✓ Legacy file also saved: terraclimate_features_training.csv (pet only)
════════════════════════════════════════════════════════════════════════════════


In [27]:
# Preview File
Terraclimate_training_df.head()

,Latitude,Longitude,Sample Date,pet,ppt,soil,tmax,tmin
0,-28.760833,17.730278,02-01-2011,174.199997,32.7,0.0,36.099998,22.689999
1,-26.861111,28.884722,03-01-2011,124.099998,51.1,12.8,27.160000,13.219999
2,-26.450000,28.085833,03-01-2011,127.500000,62.7,6.8,27.519999,14.090000
3,-27.671111,27.236944,03-01-2011,129.699997,84.2,7.2,28.869999,14.639999
4,-27.356667,27.286389,03-01-2011,129.199997,78.0,7.8,28.670000,14.690000


---

### Final Step: Combine and Save Files

**After running all 5 extraction cells above**, run the cells below to:
1. Combine training variables → Save `terraclimate_features_training_EXTENDED.csv`
2. Combine validation variables → Save `terraclimate_features_validation_EXTENDED.csv`

Both files will contain all 5 climate variables: pet, ppt, soil, tmax, tmin

---

In [28]:
# ══════════════════════════════════════════════════════════════════════════════
# COMBINE ALL VARIABLES AND SAVE VALIDATION FILE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 80)
print("  COMBINING ALL VARIABLES - VALIDATION DATASET")
print("═" * 80)

# Combine all validation variables
print("\n[1/3] Combining all variables into single DataFrame...")
Terraclimate_validation_df = pd.concat([
    pet_validation_mapped,
    ppt_validation_mapped,
    soil_validation_mapped,
    tmax_validation_mapped,
    tmin_validation_mapped
], axis=1)

print(f"      ✓ Combined shape: {Terraclimate_validation_df.shape}")
print(f"      ✓ Columns: {list(Terraclimate_validation_df.columns)}")

# Add location and date information
print("\n[2/3] Adding location and date columns...")
Terraclimate_validation_df['Latitude'] = Validation_df['Latitude'].values
Terraclimate_validation_df['Longitude'] = Validation_df['Longitude'].values
Terraclimate_validation_df['Sample Date'] = Validation_df['Sample Date'].values

# Reorder columns to match training file
Terraclimate_validation_df = Terraclimate_validation_df[column_order]

# Save EXTENDED validation file
val_output_file = 'terraclimate_features_validation_EXTENDED.csv'
Terraclimate_validation_df.to_csv(val_output_file, index=False)

print(f"\n[3/3] ✅ VALIDATION FILE SAVED: {val_output_file}")
print(f"      Shape: {Terraclimate_validation_df.shape}")
print(f"      Size: {os.path.getsize(val_output_file) / 1_000_000:.2f} MB")

# Also save legacy single-variable file for backward compatibility
Terraclimate_validation_df[['Latitude', 'Longitude', 'Sample Date', 'pet']].to_csv(
    'terraclimate_features_validation.csv', index=False
)
print(f"      ✓ Legacy file also saved: terraclimate_features_validation.csv (pet only)")

print("\n" + "█" * 80)
print("  ✅ EXTRACTION COMPLETE!")
print("█" * 80)
print("\nFiles created:")
print(f"  1. {output_file} ({Terraclimate_training_df.shape[0]} rows × {Terraclimate_training_df.shape[1]} cols)")
print(f"  2. {val_output_file} ({Terraclimate_validation_df.shape[0]} rows × {Terraclimate_validation_df.shape[1]} cols)")
print("\nNext steps:")
print("  → Use these EXTENDED files in Optimization 10")
print("  → Expected columns: Latitude, Longitude, Sample Date, pet, ppt, soil, tmax, tmin")
print("═" * 80)


════════════════════════════════════════════════════════════════════════════════
  COMBINING ALL VARIABLES - VALIDATION DATASET
════════════════════════════════════════════════════════════════════════════════

[1/3] Combining all variables into single DataFrame...
      ✓ Combined shape: (200, 5)
      ✓ Columns: ['pet', 'ppt', 'soil', 'tmax', 'tmin']

[2/3] Adding location and date columns...

[3/3] ✅ VALIDATION FILE SAVED: terraclimate_features_validation_EXTENDED.csv
      Shape: (200, 8)
      Size: 0.01 MB
      ✓ Legacy file also saved: terraclimate_features_validation.csv (pet only)

████████████████████████████████████████████████████████████████████████████████
  ✅ EXTRACTION COMPLETE!
████████████████████████████████████████████████████████████████████████████████

Files created:
  1. terraclimate_features_training_EXTENDED.csv (9319 rows × 8 cols)
  2. terraclimate_features_validation_EXTENDED.csv (200 rows × 8 cols)

Next steps:
  → Use these EXTENDED files in Optimization

In [29]:
# Preview File
Terraclimate_validation_df.head()

,Latitude,Longitude,Sample Date,pet,ppt,soil,tmax,tmin
0,-32.043333,27.822778,01-09-2014,161.900009,25.6,4.8,28.750000,15.929999
1,-33.329167,26.077500,16-09-2015,177.600006,11.6,2.7,29.340000,15.339999
2,-32.991639,27.640028,07-05-2015,158.400009,21.5,12.0,27.389999,17.869999
3,-34.096389,24.439167,07-02-2012,130.000000,28.2,16.4,23.769999,16.070000
4,-32.000556,28.581667,01-10-2014,152.500000,27.2,5.1,27.799999,16.559999


In [30]:
# Check if data exists in memory
print(f"Validation df shape: {Terraclimate_validation_df.shape}")
Terraclimate_validation_df.head()

Validation df shape: (200, 8)


,Latitude,Longitude,Sample Date,pet,ppt,soil,tmax,tmin
0,-32.043333,27.822778,01-09-2014,161.900009,25.6,4.8,28.750000,15.929999
1,-33.329167,26.077500,16-09-2015,177.600006,11.6,2.7,29.340000,15.339999
2,-32.991639,27.640028,07-05-2015,158.400009,21.5,12.0,27.389999,17.869999
3,-34.096389,24.439167,07-02-2012,130.000000,28.2,16.4,23.769999,16.070000
4,-32.000556,28.581667,01-10-2014,152.500000,27.2,5.1,27.799999,16.559999


In [31]:
# Run this to verify all validation data exists:
print("Validation variables status:")
print(f"  pet_validation_mapped: {len(pet_validation_mapped) if 'pet_validation_mapped' in locals() else 'MISSING'}")
print(f"  ppt_validation_mapped: {len(ppt_validation_mapped) if 'ppt_validation_mapped' in locals() else 'MISSING'}")
print(f"  soil_validation_mapped: {len(soil_validation_mapped) if 'soil_validation_mapped' in locals() else 'MISSING'}")
print(f"  tmax_validation_mapped: {len(tmax_validation_mapped) if 'tmax_validation_mapped' in locals() else 'MISSING'}")
print(f"  tmin_validation_mapped: {len(tmin_validation_mapped) if 'tmin_validation_mapped' in locals() else 'MISSING'}")

Validation variables status:
  pet_validation_mapped: 200
  ppt_validation_mapped: 200
  soil_validation_mapped: 200
  tmax_validation_mapped: 200
  tmin_validation_mapped: 200


In [26]:
# --- Extract AET (Actual Evapotranspiration) ---
print("Extracting AET...")
ds_aet = load_terraclimate_dataset()
aet_climate_df = filterg(ds_aet, 'aet')
# Training
train_aet = assign_nearest_climate(Water_Quality_df, aet_climate_df, 'aet')
# Validation
val_aet = assign_nearest_climate(Validation_df, aet_climate_df, 'aet')

Extracting AET...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [02:09<00:00,  2.15s/it]


Filtering for aet completed


Mapping AET values: 100%|██████████| 200/200 [00:13<00:00, 14.33it/s]


In [27]:
# --- Extract DEF (Climate Water Deficit) ---
print("Extracting DEF...")
ds_def = load_terraclimate_dataset()
def_climate_df = filterg(ds_def, 'def')
# Training
train_def = assign_nearest_climate(Water_Quality_df, def_climate_df, 'def')
# Validation
val_def = assign_nearest_climate(Validation_df, def_climate_df, 'def')

Extracting DEF...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [01:49<00:00,  1.83s/it]


Filtering for def completed


Mapping DEF values: 100%|██████████| 200/200 [00:13<00:00, 14.79it/s]


In [28]:
# --- Extract SRAD (Downward Surface Shortwave Radiation) ---
print("Extracting SRAD...")
ds_srad = load_terraclimate_dataset()
srad_climate_df = filterg(ds_srad, 'srad')
# Training
train_srad = assign_nearest_climate(Water_Quality_df, srad_climate_df, 'srad')
# Validation
val_srad = assign_nearest_climate(Validation_df, srad_climate_df, 'srad')

Extracting SRAD...
Diagnostics: {'lat_name': 'lat', 'lon_name': 'lon', 'lat_range': (-89.97916666666664, 89.97916666666667), 'lon_range': (-179.97916666666666, 179.97916666666666), 'subset_sizes': {'time': 60, 'lat': 323, 'lon': 428}}


100%|██████████| 60/60 [02:30<00:00,  2.51s/it]


Filtering for srad completed


Mapping SRAD values: 100%|██████████| 200/200 [00:11<00:00, 16.96it/s]


In [30]:
# ══════════════════════════════════════════════════════════════════════════════
# SMART MERGE: COMBINE EXISTING EXTENDED FILES + NEW VARIABLES (AET, DEF, SRAD)
# ══════════════════════════════════════════════════════════════════════════════
print("🔧 SMART COMBINATION: Adding 3 new variables to existing EXTENDED files")
print("="*80)

# Check what variables we have in memory vs what we need to load from files
available_vars = {
    'AET': ('train_aet' in locals() and 'val_aet' in locals()),
    'DEF': ('train_def' in locals() and 'val_def' in locals()),
    'SRAD': ('train_srad' in locals() and 'val_srad' in locals())
}

print("📊 MEMORY STATUS CHECK:")
for var, available in available_vars.items():
    status = "✅" if available else "❌"
    print(f"   {var}: {status}")

# Load existing EXTENDED files (contains first 5 variables: PET, PPT, SOIL, TMAX, TMIN)
print(f"\n[1/4] Loading existing EXTENDED CSV files...")
try:
    existing_training = pd.read_csv('terraclimate_features_training_EXTENDED.csv')
    existing_validation = pd.read_csv('terraclimate_features_validation_EXTENDED.csv')
    
    print(f"   ✅ Training EXTENDED loaded: {existing_training.shape}")
    print(f"   ✅ Validation EXTENDED loaded: {existing_validation.shape}")
    print(f"   📋 Existing columns: {list(existing_training.columns)}")
    
except FileNotFoundError as e:
    print(f"   ❌ Could not load existing files: {e}")
    print("   💡 Please ensure the EXTENDED files exist before running this cell.")
    raise

# Verify the 3 new variables are available in memory
print(f"\n[2/4] Verifying new variables in memory...")
if not all(available_vars.values()):
    missing = [var for var, avail in available_vars.items() if not avail]
    print(f"   ❌ Missing variables: {missing}")
    print("   💡 Please run the AET, DEF, SRAD extraction cells first.")
    raise ValueError(f"Missing required variables: {missing}")

print(f"   ✅ All 3 new variables available in memory")
print(f"       AET: train_aet ({len(train_aet)}), val_aet ({len(val_aet)})")
print(f"       DEF: train_def ({len(train_def)}), val_def ({len(val_def)})")  
print(f"       SRAD: train_srad ({len(train_srad)}), val_srad ({len(val_srad)})")

# Combine existing data with new variables
print(f"\n[3/4] Combining data...")

# Training: Existing + 3 new variables
train_final = pd.concat([
    existing_training,  # Has: Latitude, Longitude, Sample Date, pet, ppt, soil, tmax, tmin
    train_aet,         # Add: aet
    train_def,         # Add: def  
    train_srad         # Add: srad
], axis=1)

# Validation: Existing + 3 new variables
val_final = pd.concat([
    existing_validation,  # Has: Latitude, Longitude, Sample Date, pet, ppt, soil, tmax, tmin
    val_aet,             # Add: aet
    val_def,             # Add: def
    val_srad             # Add: srad
], axis=1)

print(f"   ✅ Training combined: {train_final.shape}")
print(f"   ✅ Validation combined: {val_final.shape}")

# Save final files
print(f"\n[4/4] Saving final datasets...")
train_filename = 'terraclimate_features_training_final.csv'
val_filename = 'terraclimate_features_validation_final.csv'

train_final.to_csv(train_filename, index=False)
val_final.to_csv(val_filename, index=False)

# Final Report
print("="*80)
print("🎉 SUCCESS! FINAL TERRACLIMATE DATASETS CREATED")
print("="*80)

print(f"\n📁 TRAINING FILE: {train_filename}")
print(f"   Shape: {train_final.shape[0]} rows × {train_final.shape[1]} columns")
print(f"   Size: {os.path.getsize(train_filename) / (1024*1024):.2f} MB")

print(f"\n📁 VALIDATION FILE: {val_filename}")
print(f"   Shape: {val_final.shape[0]} rows × {val_final.shape[1]} columns")
print(f"   Size: {os.path.getsize(val_filename) / (1024*1024):.2f} MB")

print(f"\n🌍 COMPLETE VARIABLE SET (11 columns total):")
print(f"   📍 Location & Time: Latitude, Longitude, Sample Date")
print(f"   🌡️  Original 5 variables: pet, ppt, soil, tmax, tmin")
print(f"   ⚡ New 3 variables: aet, def, srad")
print(f"   📋 Full columns: {list(train_final.columns)}")

print(f"\n📊 DATA PREVIEW:")
display(train_final.head())

🔧 SMART COMBINATION: Adding 3 new variables to existing EXTENDED files
📊 MEMORY STATUS CHECK:
   AET: ✅
   DEF: ✅
   SRAD: ✅

[1/4] Loading existing EXTENDED CSV files...
   ✅ Training EXTENDED loaded: (9319, 8)
   ✅ Validation EXTENDED loaded: (200, 8)
   📋 Existing columns: ['Latitude', 'Longitude', 'Sample Date', 'pet', 'ppt', 'soil', 'tmax', 'tmin']

[2/4] Verifying new variables in memory...
   ✅ All 3 new variables available in memory
       AET: train_aet (9319), val_aet (200)
       DEF: train_def (9319), val_def (200)
       SRAD: train_srad (9319), val_srad (200)

[3/4] Combining data...
   ✅ Training combined: (9319, 11)
   ✅ Validation combined: (200, 11)

[4/4] Saving final datasets...
🎉 SUCCESS! FINAL TERRACLIMATE DATASETS CREATED

📁 TRAINING FILE: terraclimate_features_training_final.csv
   Shape: 9319 rows × 11 columns
   Size: 0.77 MB

📁 VALIDATION FILE: terraclimate_features_validation_final.csv
   Shape: 200 rows × 11 columns
   Size: 0.02 MB

🌍 COMPLETE VARIABLE SET

,Latitude,Longitude,Sample Date,pet,ppt,soil,tmax,tmin,aet,def,srad
0,-28.760833,17.730278,02-01-2011,174.2,32.7,0.0,36.100000,22.689999,31.100000,143.100006,270.200714
1,-26.861111,28.884722,03-01-2011,124.1,51.1,12.8,27.160000,13.219999,53.799999,70.300003,227.500427
2,-26.450000,28.085833,03-01-2011,127.5,62.7,6.8,27.519999,14.090000,60.799999,66.700005,230.899384
3,-27.671111,27.236944,03-01-2011,129.7,84.2,7.2,28.869999,14.639999,83.200005,46.500000,220.299393
4,-27.356667,27.286389,03-01-2011,129.2,78.0,7.8,28.670000,14.690000,77.300003,51.900002,222.899994


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DOWNLOAD SOLUTION FOR CSV FILES
# ═══════════════════════════════════════════════════════════════════════════════

import os
import glob
from IPython.display import Javascript, display

print("🔍 Checking for available CSV files...")

# Check what CSV files exist in the current directory
csv_files = glob.glob("*.csv")
landsat_files = [f for f in csv_files if 'landsat' in f.lower()]
terraclimate_files = [f for f in csv_files if 'terraclimate' in f.lower()]

print(f"\n📊 Found {len(csv_files)} total CSV files:")
print(f"   🛰️  Landsat files: {len(landsat_files)}")
print(f"   🌍 TerraClimate files: {len(terraclimate_files)}")

if csv_files:
    print("\n📁 Available files:")
    for file in sorted(csv_files):
        size_mb = os.path.getsize(file) / (1024*1024)
        print(f"   • {file} ({size_mb:.1f} MB)")
    
    print("\n" + "="*60)
    print("💡 DOWNLOAD OPTIONS:")
    print("="*60)
    print("Option 1: Right-click on files in VS Code file explorer → 'Download'")
    print("Option 2: Use the Files panel on the left to download")
    print("Option 3: If in Jupyter, files are in the same directory as this notebook")
    print("Option 4: Copy files to your local machine via terminal/command line")
    
else:
    print("\n⚠️  No CSV files found. Please run the extraction cells first.")

# Also provide a file listing command for terminal use
print(f"\n🔧 Terminal command to list files:")
print(f"   ls -lh *.csv")